<a href="https://colab.research.google.com/github/andrewgodbout/MCS-3950-F26/blob/main/Notebooks/L02-Digital-Images/L2_Digital_Images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 2 — Digital Images: Follow-Along Examples
**MCS 3950 Computer Vision · UPEI · Fall 2026**

This notebook accompanies **Lecture 2**

| Section | Topic |
|---------|-------|
| 1 | Loading & viewing an image |
| 2 | Bit depth & quantization |
| 3 | Image shape & channels |
| 4 | NumPy indexing |
| 5 | Point operations & the overflow trap |
| 6 | Colour models: RGB, HSV, CIE L\*a\*b\* |
| ✏️ | **Try at home** |



In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
# Run this cell first.  On Colab everything is pre-installed; locally you may
# need:  pip install opencv-python scikit-image matplotlib numpy
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from skimage import data as skdata

# Consistent figure style
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("OpenCV  :", cv2.__version__)
print("NumPy   :", np.__version__)


---
## 1 · Loading & Viewing an Image


We'll use a built-in sample image (`skimage.data.astronaut`) throughout this
notebook — it's colourful and has a mix of skin tones, fabric, and background,
making it ideal for demonstrating colour operations.

> **Slides connection:** every pixel in this image is a point on the raster
> grid.  The pixel grid visualisation in the powerpoint slides shows the
> same idea at extreme zoom.


In [ ]:
# Load the sample image — already in RGB (unlike cv2.imread which gives BGR)
img_rgb = skdata.astronaut()          # uint8, shape (512, 512, 3)

plt.figure(figsize=(5, 5))
plt.imshow(img_rgb)
plt.title(f"astronaut.png  |  shape = {img_rgb.shape}  |  dtype = {img_rgb.dtype}")
plt.axis("off")
plt.tight_layout()
plt.show()


### ⚠️ The OpenCV BGR Gotcha

`cv2.imread()` loads images in **BGR** order (Blue, Green, Red), not RGB.
Matplotlib's `imshow()` expects RGB, so displaying a BGR image directly
produces wrong colours.  Run the cell below to see the effect.


In [ ]:
# Simulate what happens when you forget to convert from BGR → RGB
img_bgr_sim = img_rgb[:, :, ::-1]    # flip channel order to mimic cv2.imread output

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_bgr_sim)
axes[0].set_title("Displayed as-is (BGR order)\n← wrong colours!")
axes[1].imshow(img_rgb)
axes[1].set_title("Channels flipped back (RGB)\n← correct")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Fix options:")
print("  cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)   # convert")
print("  img_bgr[:, :, ::-1]                         # reverse slice")


> **Remember:** if you load images with `cv2.imread()` in your project code,
> always convert before displaying with Matplotlib.  Skipping this step is the
> #1 source of baffling colour errors in student assignments.


---
## 2 · Bit Depth & Quantization


the gradient bars from the powerpoint showed how increasing
bit depth produces smoother tones.  Here we reproduce that effect on a real
grayscale image.


In [ ]:
# Convert to grayscale for this section
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

print(f"dtype : {img_gray.dtype}")
print(f"shape : {img_gray.shape}")
print(f"min   : {img_gray.min()}   max: {img_gray.max()}")
print(f"memory: {img_gray.nbytes / 1024:.1f} KB  (uncompressed)")


### Simulating Lower Bit Depths

We can simulate what the image would look like with fewer bits by reducing
the number of distinct intensity levels.  A 2-bit image has only 4 levels;
notice the harsh "banding" (posterization) that appears.


In [ ]:
def quantize(img_gray, bits):
    """Reduce a uint8 grayscale image to the given number of bits."""
    levels  = 2 ** bits
    step    = 256 // levels          # spacing between levels
    q = (img_gray // step) * step    # snap each pixel to nearest level
    return q.astype(np.uint8)

bit_depths = [2, 4, 8]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, bits in zip(axes, bit_depths):
    ax.imshow(quantize(img_gray, bits), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{bits}-bit  ({2**bits} levels)")
    ax.axis("off")
plt.suptitle("Effect of Bit Depth on Image Quality", y=1.01)
plt.tight_layout()
plt.show()


**Observation:** the transition from 4-bit to 8-bit is dramatic; beyond 8
bits the improvement becomes imperceptible to the human eye under most
conditions.  That's why consumer cameras standardised on 8-bit per channel.


---
## 3 · Image Shape & Channels


a colour image is three stacked grayscale images —
one per channel.  Let's pull the stack apart.


In [ ]:
H, W, C = img_rgb.shape
print(f"Height (rows)  : {H} px")
print(f"Width  (cols)  : {W} px")
print(f"Channels       : {C}  (Red, Green, Blue)")
print(f"Total pixels   : {H * W:,}")
print(f"Memory (uint8) : {img_rgb.nbytes / 1024:.0f} KB  ≈  {img_rgb.nbytes / 1e6:.2f} MB")


In [ ]:
# Split into individual channels
r = img_rgb[:, :, 0]   # Red
g = img_rgb[:, :, 1]   # Green
b = img_rgb[:, :, 2]   # Blue

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(img_rgb)
axes[0].set_title("Original (RGB)")

channel_data = [(r, "Reds",  "Reds_r"),
                (g, "Green", "Greens_r"),
                (b, "Blues", "Blues_r")]
for ax, (ch, title, cmap) in zip(axes[1:], channel_data):
    ax.imshow(ch, cmap=cmap, vmin=0, vmax=255)
    ax.set_title(f"{title[0]} channel min={ch.min()}  max={ch.max()}")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


Notice that the suit (which is red-orange) is brightest in the **R channel**
and darkest in the **B channel**.  The background reflects roughly equally
across all three channels, which is what makes it appear neutral grey.

Note also: `Reds_r` (is bright for large pixel values, whereas colormap `Reds` is deep/dark red for large values), a previous notebook used `Reds, it's a good idea to watch the cmap setting as it controls how things are visualized.


---
## 4 · NumPy Indexing


In [ ]:
# ── Single pixel ─────────────────────────────────────────────────────────────
r_val, g_val, b_val = img_rgb[100, 200]      # row 100, col 200
print(f"img_rgb[100, 200]    = ({r_val}, {g_val}, {b_val})")

# ── Single channel value ──────────────────────────────────────────────────────
green_at_pixel = img_rgb[100, 200, 1]        # channel index 1 = Green
print(f"img_rgb[100, 200, 1] = {green_at_pixel}  (Green channel only)")

# ── Entire channel as 2-D array ───────────────────────────────────────────────
red_channel  = img_rgb[:, :, 0]
print(f"img_rgb[:, :, 0].shape = {red_channel.shape}  (full Red channel)")

# ── Crop ─────────────────────────────────────────────────────────────────────
crop = img_rgb[60:180, 180:320]
print(f"Crop shape = {crop.shape}")


In [ ]:
# Visualise the crop region on the original image
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: original with crop box overlaid
axes[0].imshow(img_rgb)
rect = patches.Rectangle((180, 60), 140, 120,
                          linewidth=2, edgecolor="yellow", facecolor="none")
axes[0].add_patch(rect)
axes[0].set_title("Original  (crop region in yellow)")
axes[0].axis("off")

# Right: the crop
axes[1].imshow(crop)
axes[1].set_title(f"Cropped region  {crop.shape[1]}×{crop.shape[0]} px")
axes[1].axis("off")

plt.tight_layout()
plt.show()


---
## 5 · Point Operations & The Overflow Trap


**Slide connection:** point operations transform each pixel independently.
This section demonstrates each operation from the slides — and shows exactly
what goes wrong when you forget about `uint8` overflow.


In [ ]:
# ── Invert / Negate — always safe ────────────────────────────────────────────
img_inverted = 255 - img_rgb
# 255 - 0 = 255,  255 - 255 = 0  →  always stays in [0, 255]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_rgb);    axes[0].set_title("Original");    axes[0].axis("off")
axes[1].imshow(img_inverted); axes[1].set_title("Inverted (255 − img)"); axes[1].axis("off")
plt.tight_layout(); plt.show()
print("Inversion is always safe — subtracting FROM 255 can never underflow.")


### The Overflow Bug

Now let's try brightening.  Adding a constant to a `uint8` array in NumPy
uses **modular (wrapping) arithmetic** — it does not clamp at 255.


In [ ]:
# ── WRONG: NumPy uint8 wraps around ──────────────────────────────────────────
amount = 100

# Check what happens to a pixel that's already bright
bright_pixel  = np.uint8(200)
result_uint8  = bright_pixel + np.uint8(amount)
print(f"np.uint8(200) + np.uint8({amount}) = {result_uint8}   ← expected 255, got {result_uint8}!")
print(f"Why? 200 + {amount} = {200+amount}; {200+amount} mod 256 = {(200+amount) % 256}")

# Apply to the whole image — note NO ERROR is raised
img_bright_wrong = img_rgb + np.uint8(amount)   # wraps silently

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_rgb);          axes[0].set_title("Original");    axes[0].axis("off")
axes[1].imshow(img_bright_wrong); axes[1].set_title(f"img + {amount}  ← WRONG\nDark patches where highlights wrapped"); axes[1].axis("off")
plt.tight_layout(); plt.show()


Those dark patches in the bright areas of the image are the overflow in
action.  No error, no warning — just silently wrong pixel values.

### Safe Alternatives


In [ ]:
amount = 100

# ── Safe option 1: cv2.add — saturates at 255 ────────────────────────────────
# cv2.add adds a scalar to every pixel and clamps to [0, 255]
img_bright_cv2 = cv2.add(img_rgb, amount ) # you can also supply a vector here (100, 100, 100, 0) with scales for R, G, B

# ── Safe option 2: cast to float32, operate, clip, cast back ─────────────────
img_bright_f32 = np.clip(img_rgb.astype(np.float32) + amount, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, im, title in zip(axes,
    [img_bright_wrong, img_bright_cv2, img_bright_f32],
    [f"img + {amount}  (WRONG — wraps)",
     f"cv2.add(img, {amount})  (clamps)",
     f"float32 → +{amount} → clip → uint8"]):
    ax.imshow(im); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# ── Contrast scaling — always needs float first ───────────────────────────────
scale = 1.5
img_contrast = np.clip(img_rgb.astype(np.float32) * scale, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_rgb);      axes[0].set_title("Original");         axes[0].axis("off")
axes[1].imshow(img_contrast); axes[1].set_title(f"Contrast ×{scale}"); axes[1].axis("off")
plt.tight_layout(); plt.show()


---
## 6 · Colour Models


### 6a. RGB → Grayscale: Mean vs. Luminance Weights

grayscale conversion uses perceptual weights
(0.299 R + 0.587 G + 0.114 B), **not a simple average**.  Let's see why that
matters.

Note: We emphasize green when converting to grayscale because human eyes are biologically wired to perceive green light as significantly brighter and more detailed than red or blue light.


In [ ]:
# Method 1: simple mean (wrong perceptually)
gray_mean = img_rgb.mean(axis=2).astype(np.uint8)

# Method 2: luminance weights (Rec.601)
weights   = np.array([0.299, 0.587, 0.114], dtype=np.float32)
gray_lum  = (img_rgb.astype(np.float32) @ weights).astype(np.uint8)

# Method 3: OpenCV (uses the same Rec.601 weights internally)
gray_cv   = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, g, title in zip(axes,
    [gray_mean, gray_lum, gray_cv],
    ["Mean  (equal weights 0.33 each)",
     "Luminance  (0.299R + 0.587G + 0.114B)",
     "cv2.COLOR_RGB2GRAY  (Rec.601)"]):
    ax.imshow(g, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout(); plt.show()

# Difference image — amplified so differences are visible
diff = np.abs(gray_mean.astype(int) - gray_lum.astype(int)).astype(np.uint8)
print(f"Max pixel difference (mean vs luminance): {diff.max()}")
print(f"Mean pixel difference: {diff.mean():.2f}")


The difference is subtle on this image but becomes significant on scenes
with strong red or blue content — e.g., emergency vehicle lights, traffic
signs, or seasonal foliage in the aerial imagery.


### 6b. HSV — Hue, Saturation, Value

**Slide connection:** each row of bars from the HSV slide corresponds to one
of the three channels below.


In [ ]:
# OpenCV works in BGR, so convert first
img_bgr = img_rgb[:, :, ::-1]
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

H_ch = img_hsv[:, :, 0]   # Hue: 0–179 in OpenCV (not 0–360 — watch out!)
S_ch = img_hsv[:, :, 1]   # Saturation: 0–255
V_ch = img_hsv[:, :, 2]   # Value: 0–255

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
axes[0].imshow(img_rgb); axes[0].set_title("Original (RGB)"); axes[0].axis("off")
for ax, ch, cmap, title, note in zip(axes[1:],\
    [H_ch, S_ch, V_ch],\
    ["hsv", "gray", "gray"],\
    ["H — Hue\n", "S — Saturation\n", "V — Value\n"],\
    ["0–179 (OpenCV) 0° = red, 60° = yellow, 120° = green\n",\
     "0 = grey → 255 = vivid\n",\
     "0 = black → 255 = bright"]):
    ax.imshow(ch, cmap=cmap, vmin=0, vmax=255)
    ax.set_title(f"{title}{note}")
    ax.axis("off")
plt.tight_layout(); plt.show()

print("⚠  OpenCV HSV uses H in range 0–179 (half of 360°) to fit in uint8.")
print("   Multiply by 2 to get standard degrees, or keep in mind when thresholding.")


### 6c. CIE L\*a\*b\* — Perceptual Colour Space

**Slide connection:** L\* is pure lightness (independent of colour);
a\* encodes green↔red; b\* encodes blue↔yellow.


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

L_ch = img_lab[:, :, 0]   # Lightness: 0–255 in OpenCV's uint8 encoding
a_ch = img_lab[:, :, 1]   # a*: 0–255 (128 = neutral, <128 = green, >128 = red)
b_ch = img_lab[:, :, 2]   # b*: 0–255 (128 = neutral, <128 = blue, >128 = yellow)

cmap_b = LinearSegmentedColormap.from_list("Lab_b", ["#0000FF", "#808080", "#FFFF00"])

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
axes[0].imshow(img_rgb); axes[0].set_title("Original (RGB)"); axes[0].axis("off")
for ax, ch, cmap, title in zip(axes[1:],
    [L_ch, a_ch, b_ch],
    ["gray", "PiYG_r", cmap_b],
    ["L* — Lightness (device-independent)",
     "a* — Green ← · → Red (128 = neutral)",
     "b* — Blue ← · → Yellow (128 = neutral)"]):
    ax.imshow(ch, cmap=cmap, vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout(); plt.show()

print("Note: OpenCV encodes L* in [0,255] and a*, b* offset by 128.")
print("      True Lab values: L in [0,100]; a, b in roughly [−128, +127].")


**Why does this matter?**

If you are doing change detection between images,
computing pixel differences in L\*a\*b\* gives you *perceptually meaningful*
change values — a ΔE distance in Lab space corresponds to how different two
colours actually look to a human observer.  The same numerical difference in
RGB can be nearly invisible or very salient depending on the colours involved.


---
## Color Similarity

How similar are two colors?

But first a quick test. Can you detect small changes in green hues?

In [ ]:
from skimage import color as skcolor

def make_swatch_grid(base_hsv, hue_shift, rows=3, cols=4, swatch_px=90, gap_px=16, bg_gray=190):
    """Build a grid of colour swatches; one swatch (chosen at random) has its hue
    shifted by `hue_shift` degrees relative to the rest.

    base_hsv   : (H, S, V) tuple, H in degrees [0-360), S and V in [0, 1]
    hue_shift  : degrees to shift the odd-one-out swatch's hue by
    gap_px     : width (in pixels) of the neutral-gray gap surrounding each swatch.
                 Bigger gap => swatches can no longer be compared edge-to-edge,
                 which makes the odd-one-out noticeably harder to spot.
    bg_gray    : brightness (0-255) of the gap/background — kept neutral gray so it
                 doesn't itself bias the color comparison.
    Returns    : (grid_rgb, odd_index) — grid_rgb is a uint8 image, odd_index is
                 the flat index (row-major) of the outlier swatch.
    """
    n = rows * cols
    odd_index = np.random.randint(n)
    h0, s0, v0 = base_hsv

    cell_px = swatch_px + gap_px
    grid = np.full(
        (rows * cell_px + gap_px, cols * cell_px + gap_px, 3), bg_gray, dtype=np.uint8
    )
    for i in range(n):
        r, c = divmod(i, cols)
        h = (h0 + hue_shift) % 360 if i == odd_index else h0
        # HSV -> RGB (OpenCV expects H in [0,179], S,V in [0,255])
        hsv_px = np.uint8([[[h / 2, s0 * 255, v0 * 255]]])
        rgb_px = cv2.cvtColor(hsv_px, cv2.COLOR_HSV2RGB)[0, 0]
        y0 = gap_px + r * cell_px
        x0 = gap_px + c * cell_px
        grid[y0:y0 + swatch_px, x0:x0 + swatch_px] = rgb_px

    return grid, odd_index


In [ ]:
#  - 'easy' grid: a clearly different hue (most people spot it instantly)
#  - 'hard' grid: a small hue shift inside the blue-green region

np.random.seed() # add a seed here to produce the same random position each iteration

easy_grid, easy_idx = make_swatch_grid(base_hsv=(140, 0.55, 0.75), hue_shift=45)   # green -> teal
hard_grid, hard_idx = make_swatch_grid(base_hsv=(150, 0.55, 0.75), hue_shift=7)    # small green->blue-green nudge

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(easy_grid); axes[0].set_title('"Easy" — large hue shift'); axes[0].axis('off')
axes[1].imshow(hard_grid); axes[1].set_title('"Hard" — small hue shift'); axes[1].axis('off')
plt.suptitle('Spot the odd swatch — try both before running the reveal cell below', fontsize=12)
plt.tight_layout(); plt.show()

print('Take a guess for each grid (row-major index, 0-11) before revealing the answer below.')

In [ ]:
# Reveal — how'd you do?
print(f'Easy grid — odd swatch was at index {easy_idx}')
print(f'Hard grid — odd swatch was at index {hard_idx}')

---
Instead of relying on your own eyes, let's measure color difference numerically — the same way a CV pipeline would. The obvious approach is Euclidean distance in RGB:

$$d_{RGB}(c_1, c_2) = \sqrt{(R_1-R_2)^2 + (G_1-G_2)^2 + (B_1-B_2)^2}$$

The problem: RGB space is **not perceptually uniform**. Equal RGB distances do not correspond to equal *perceived* differences — a shift of 10 units might be invisible in one part of the color cube and obvious in another. **CIE Lab** was specifically engineered to fix this: it's built from human color-matching experiments so that Euclidean distance in Lab space (often called ΔE) tracks perceived difference much more closely.

In [ ]:
def rgb_distance(c1, c2):
    """Euclidean distance between two RGB colours, each a length-3 array in [0,255]."""
    c1, c2 = np.asarray(c1, dtype=float), np.asarray(c2, dtype=float)
    return np.sqrt(np.sum((c1 - c2) ** 2))

def lab_distance(c1, c2):
    """Euclidean distance between two RGB colours after converting to CIE Lab (a rough delta-E)."""
    c1 = skcolor.rgb2lab(np.uint8([[c1]]))[0, 0]
    c2 = skcolor.rgb2lab(np.uint8([[c2]]))[0, 0]
    return np.sqrt(np.sum((c1 - c2) ** 2))

print('rgb_distance() and lab_distance() ready.')


In [ ]:
# Compare both metrics on a handful of colour pairs, including a blue/green boundary case
# and a pair that's RGB-close but perceptually distinct.

def hsv_to_rgb(h, s, v):
    px = np.uint8([[[h / 2, s * 255, v * 255]]])
    return cv2.cvtColor(px, cv2.COLOR_HSV2RGB)[0, 0]

pairs = {
    'clearly different (red vs blue)':      (hsv_to_rgb(0, 0.8, 0.8),   hsv_to_rgb(220, 0.8, 0.8)),
    'blue/green boundary (small shift)':     (hsv_to_rgb(150, 0.55, 0.75), hsv_to_rgb(158, 0.55, 0.75)),
    'two shades of green':   (hsv_to_rgb(120, 0.55, 0.75), hsv_to_rgb(132, 0.55, 0.75)),
    'dark vs light of the same hue':         (hsv_to_rgb(210, 0.7, 0.3),  hsv_to_rgb(210, 0.7, 0.9)),
}

print(f'{"Pair":38s} {"RGB dist":>10s} {"Lab dist":>10s}')
print('-' * 60)
swatches = []
for name, (c1, c2) in pairs.items():
    d_rgb = rgb_distance(c1, c2)
    d_lab = lab_distance(c1, c2)
    swatches.append((name, c1, c2, d_rgb, d_lab))
    print(f'{name:38s} {d_rgb:10.1f} {d_lab:10.1f}')


In [ ]:
# Visualise the pairs alongside their distances
fig, axes = plt.subplots(len(swatches), 1, figsize=(6, 1.1 * len(swatches)))
for ax, (name, c1, c2, d_rgb, d_lab) in zip(axes, swatches):
    strip = np.zeros((1, 2, 3), dtype=np.uint8)
    strip[0, 0], strip[0, 1] = c1, c2
    ax.imshow(strip, extent=[0, 2, 0, 1], aspect='auto')
    ax.set_title(f'{name}   —   RGB={d_rgb:.0f}   Lab={d_lab:.0f}', fontsize=9, loc='left')
    ax.axis('off')
plt.tight_layout(); plt.show()

---
## ✏️ Try at Home — Safe Brightness Adjustment


**Estimated time: 5–10 minutes**

Complete the `safe_adjust()` function below.  It should change the brightness
of an image by `amount` (positive = brighter, negative = darker) **without**
introducing overflow or underflow artefacts.

**Requirements:**
1. Cast the image to `float32` before arithmetic.
2. Add `amount` to every pixel value.
3. Clip the result to `[0, 255]` so no value goes out of range.
4. Cast back to `uint8` before returning.

Once implemented, the test cell will display a 5-panel strip from very dark
to very bright — all transitions should be smooth with no wrap-around patches.


In [ ]:
def safe_adjust(img, amount):
    """
    Adjust the brightness of a uint8 image by `amount`.

    Parameters
    ----------
    img    : np.ndarray, dtype uint8, shape (H, W) or (H, W, C)
    amount : numeric  — pixels brightened if positive, darkened if negative

    Returns
    -------
    np.ndarray, dtype uint8, same shape as img
    """
    # ── YOUR CODE HERE ────────────────────────────────────────────────────────
    pass   # replace this line


In [ ]:
# ── Test cell — run this after completing safe_adjust() ──────────────────────
DATA_BASE = ("https://raw.githubusercontent.com/andrewgodbout/"
             "MCS-3950-F26/main/Notebooks/data/")

def load_test_image():
    """Fetch the test image; fall back to a built-in if the network is down."""
    try:
        from skimage import io
        return io.imread(DATA_BASE + "l2_brightness_test.jpg")
    except Exception as e:
        print(f"Could not fetch the test image ({type(e).__name__}). "
              f"Falling back to the built-in coffee image.")
        return skdata.coffee()

img_test = load_test_image()

amounts = [-100, -50, 0, 50, 100]
labels  = [f"{'+' if a >= 0 else ''}{a}" for a in amounts]

fig, axes = plt.subplots(1, len(amounts), figsize=(16, 3.5))
for ax, amount, label in zip(axes, amounts, labels):
    result = img_test.copy() if amount == 0 else safe_adjust(img_test, amount)
    if result is None:
        ax.text(0.5, 0.5, "Not implemented\nyet", ha="center", va="center",
                transform=ax.transAxes, fontsize=13, color="grey")
        ax.set_facecolor("#f0f0f0")
    else:
        ax.imshow(result)
    ax.set_title(f"amount = {label}")
    ax.axis("off")

plt.suptitle("safe_adjust() — Brightness Strip", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


### What to look for

✓ **Correct:** the bright regions (+100) should appear uniformly washed out
  towards white — no dark patches anywhere.

✗ **Wrong (overflow):** if you see random dark blotches in the +50 or +100
  panels, the values are wrapping rather than clamping.  Check that you
  clipped to `[0, 255]` before casting back to `uint8`.

---

### More

Try these extensions if you want an extra challenge (code blocks below):

1. **Gamma correction** — instead of adding a constant, apply:
   `output = 255 × (input / 255) ** gamma`.  Try `gamma = 0.5` (brightens
   midtones without clipping highlights) and `gamma = 2.0` (darkens).  How
   does this differ from the additive approach?

2. **Per-channel adjustment** — modify `safe_adjust()` to accept a
   *list* of three amounts `[r_amount, g_amount, b_amount]` so you can shift
   individual colour channels.  What does `safe_adjust(img, [0, -50, 0])`
   look like?  Can you make the image look like sunset time of day?


In [ ]:
def safe_adjust_gamma(img, gamma):
    """
    Adjust the brightness of a uint8 image by gamma correction.

    Parameters
    ----------
    img    : np.ndarray, dtype uint8, shape (H, W) or (H, W, C)
    amount : numeric  — pixels brightened if positive, darkened if negative

    Returns
    -------
    np.ndarray, dtype uint8, same shape as img
    """
    return img
    pass

def safe_adjust_per_channel(img, amts):
    """
    Adjust the brightness of a uint8 image by `[a1, a2, a3]` per channel.

    Parameters
    ----------
    img    : np.ndarray, dtype uint8, shape (H, W) or (H, W, C)
    amount : np.ndarray, dtype unit8, shape (1,3)  — pixels brightened if positive, darkened if negative

    Returns
    -------
    np.ndarray, dtype uint8, same shape as img
    """
    return img
    pass


gamma = 0.5
result1 = safe_adjust_gamma(img_test, gamma)
result2 = safe_adjust_per_channel(img_test, np.array([0,-50,0]))


fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

axes[0].imshow(img_test.copy())
axes[0].set_title('orig')
axes[1].imshow(result1)
axes[1].set_title('gamma')
axes[2].imshow(result2)
axes[2].set_title('per channel')

plt.suptitle("safe_adjust() — gamma and per channel", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
